# Project Notebook: Addressing Food Insecurity

## Section 1: Problem and Population



The problem addresses food insecurity among low-income working families in high-cost cities like San Jose and Santa Clara, where even employed individuals struggle to access consistently nutritious food due to inflation. This issue aligns with UNSDG 2 (Zero Hunger) and 11 (Sustainable Cities and Communities). Vina, a 20-year-old living in Santa Clara, exemplifies the affected population; despite working part-time and supporting her family, she faces challenges in finding reliable food assistance due to unstable income and high living costs. The exact failure point is the lack of accessibility to food assistance information, which is scattered across numerous, often outdated, websites, making it difficult for individuals like Vina to quickly find relevant resources based on location and eligibility. This project remains consistent with the initial problem identification, focusing on improving information accessibility.

## Section 2: Proposed System

**Input:** A user (e.g., Vina) provides their location (e.g., zip code, current address), household size, income level, and specific food needs or preferences via a web interface or mobile application.

**AI Processing:** The system leverages AI to search, filter, and cross-reference a dynamic database of food assistance programs (food banks, meal programs, SNAP resources). It uses natural language processing (NLP) to understand user queries and match them with available resources, considering eligibility criteria, operating hours, and inventory. Machine learning models could predict the most suitable options and prioritize up-to-date information by identifying and ranking reliable data sources and flagging outdated listings.

**Output:** The system presents a curated, localized, and up-to-date list of available food assistance options. This output includes detailed information such as addresses, directions, contact numbers, eligibility requirements, opening hours, and potentially real-time availability or queue times, displayed on a map or as a navigable list.

**Real-World Action:** Vina receives immediate, actionable information, allowing her to efficiently navigate to the nearest and most appropriate food resource. This reduces the time and effort spent searching, increases access to nutritious food, and alleviates the stress associated with food insecurity for her and her family.

## Section 3: Project Code

This section demonstrates key components of the proposed system by adapting code from existing labs. Note: For this submission, generic code examples are provided to illustrate the *type* of functionality, and would be replaced with specific lab code implementations in a full project.

### Lab Adaptation 1: Text Generation

Text generation is a foundational capability of modern language models: given a prompt, the model produces a written response. In the context of addressing food insecurity, this means drafting empathetic and informative responses to residents like Vina, no operator required. This lab demonstrates how to use the `get_completion` helper function with Google's Gemini 1.5 Flash model to generate acknowledgment letters, focusing on how the `food_assistance_system_prompt` is designed to serve multilingual residents.


In [1]:
import google.generativeai as genai
from google.colab import userdata
import time

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
print("Gemini initialized successfully.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini initialized successfully.


In [2]:
import google.generativeai as genai
from google.colab import userdata
import time
from google.api_core.exceptions import TooManyRequests
import re

def get_completion(prompt, system_instruction="You are a helpful assistant.", max_retries=3):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=system_instruction
    )

    retries = 0
    while retries < max_retries:
        try:
            response = m.generate_content(prompt)
            time.sleep(12)  # stays under free tier rate limit (5 requests/min)
            return response.text
        except TooManyRequests as e:
            retries += 1
            if retries == max_retries:
                raise e # Re-raise if max retries reached

            # Extract retry time from the error message
            match = re.search(r'Please retry in (\d+\.?\d*)s\.', str(e))
            wait_time = 60 # Default wait time if not found
            if match:
                wait_time = float(match.group(1))

            print(f"Quota exceeded. Retrying in {wait_time:.2f} seconds... (Attempt {retries}/{max_retries})")
            time.sleep(wait_time + 1) # Add a small buffer
        except Exception as e:
            # Handle other potential errors
            raise e

    # Should not reach here if max_retries is handled correctly
    return ""

food_assistance_system_prompt = """
You are a helpful assistant for residents seeking food assistance in San Jose and Santa Clara. Your goal is to provide clear, concise, and relevant information about food resources.

When a resident asks for help, you should:
1. Acknowledge their request empathetically.
2. Ask for their approximate location (e.g., zip code, neighborhood) to find nearby resources.
3. Briefly ask about their household size and any specific dietary needs or preferences (e.g., allergies, cultural foods).
4. Inform them that you will provide links or contact information for relevant food banks, pantries, or meal programs once you have more details. (For this demonstration, we will simulate the response, as actual resource lookup is beyond the scope of text generation alone).
5. Keep your response polite and encouraging, and under 100 words.
"""

# Example resident query
resident_query_food_assistance = (
    "I'm a working single mother of two in Santa Clara. "
    "My income is unstable, and it's hard to afford groceries for my kids. "
    "Where can I find food assistance?"
)

response_food_assistance = get_completion(resident_query_food_assistance, food_assistance_system_prompt)

print("--- Resident Query ---")
print(resident_query_food_assistance)
print("\n--- AI-drafted Food Assistance Response ---")
print(response_food_assistance)


--- Resident Query ---
I'm a working single mother of two in Santa Clara. My income is unstable, and it's hard to afford groceries for my kids. Where can I find food assistance?

--- AI-drafted Food Assistance Response ---
I understand this is a challenging time, and I want to help you find the support you need for your family.

To best assist you, could you please provide your zip code or a general neighborhood in Santa Clara? Also, are there any specific dietary needs or preferences (e.g., allergies, halal, vegetarian) for you or your children?

Once I have this information, I can help connect you to nearby food banks, pantries, and meal programs.


This initial response demonstrates how the AI can engage the user, gather necessary information, and set expectations for finding resources. In a real application, the next step would involve integrating a database of food assistance programs and a location-based search functionality.

### Lab Adaptation 2: Structured Data Extraction

Structured data extraction means constraining the AI's output to a fixed set of fields—a schema—so that every response has the same shape. In Lab Adaptation 1, the model produced free-form text. Here, we define exactly what fields are needed—such as `user_location`, `assistance_type`, `eligibility_concerns`, `urgency`, and `user_language`—and the model is prompted to fill them in consistently. This is crucial because a backend system or database cannot easily process a free-form paragraph but can readily integrate data from a structured JSON form, making the extracted information actionable for connecting individuals like Vina with relevant resources.

In [3]:
import json
resident_message = (
    "I'm Vina, a 20-year-old in Santa Clara. My family's income is unstable, and I'm looking for food banks or food resources. "
    "It's hard to find up-to-date information, and I need to know about eligibility."
)

test_messages = [
    {
        "label": "Food Insecurity (English)",
        "message": "I'm a single mom in San Jose with two kids. My income is really tight, and I'm looking for food banks or any programs that can help with groceries. We need help soon."
    },
    {
        "label": "Food Insecurity (Spanish)",
        "message": "Soy una madre soltera en San Jose con dos hijos. Mis ingresos son muy ajustados y estoy buscando bancos de alimentos o cualquier programa que pueda ayudar con los comestibles. Necesitamos ayuda pronto."
    },
    {
        "label": "Student Food Aid",
        "message": "I'm a student at Santa Clara University. I'm looking for student food assistance programs on campus or nearby. I'm planning for next semester."
    }
]

schema_prompt = """
Extract information from this food assistance request.
Return ONLY valid JSON with exactly these five fields:
{
  "user_location": string (the city or area the user is in, e.g., "Santa Clara", "San Jose"),
  "assistance_type": string (what kind of food assistance they are looking for, e.g., "food bank", "meal program", "SNAP benefits", "food resources"),
  "eligibility_concerns": string (any concerns or details related to eligibility, e.g., "income unstable", "family size", "student status"),
  "urgency": "LOW" or "MEDIUM" or "HIGH",
  "user_language": string (language the user wrote in, e.g., "English", "Spanish", "Vietnamese")
}
Urgency guide: LOW = planning for future, MEDIUM = need help soon, HIGH = immediate need for food.
No explanation. No markdown. JSON only.
"""

print("Schema defined. The AI must return exactly these five fields:")
for field in ["user_location", "assistance_type", "eligibility_concerns", "urgency", "user_language"]:
    print(f"  - {field}")

def extract_structured(message):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=schema_prompt
    )
    response = m.generate_content(message)
    time.sleep(20)  # stays under free tier rate limit
    raw = response.text.strip()
    # Strip markdown code fences if present
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return json.loads(raw)

for item in test_messages:
    print(f"=== {item['label']} ===")
    print(f"Input: {item['message']}")
    result = extract_structured(item["message"])
    print("Output:")
    print(json.dumps(result, indent=2, ensure_ascii=False))
    print()

Schema defined. The AI must return exactly these five fields:
  - user_location
  - assistance_type
  - eligibility_concerns
  - urgency
  - user_language
=== Food Insecurity (English) ===
Input: I'm a single mom in San Jose with two kids. My income is really tight, and I'm looking for food banks or any programs that can help with groceries. We need help soon.
Output:
{
  "user_location": "San Jose",
  "assistance_type": "food bank",
  "eligibility_concerns": "income tight, single mom with two kids",
  "urgency": "MEDIUM",
  "user_language": "English"
}

=== Food Insecurity (Spanish) ===
Input: Soy una madre soltera en San Jose con dos hijos. Mis ingresos son muy ajustados y estoy buscando bancos de alimentos o cualquier programa que pueda ayudar con los comestibles. Necesitamos ayuda pronto.
Output:
{
  "user_location": "San Jose",
  "assistance_type": "food bank, food resources",
  "eligibility_concerns": "income unstable, family size",
  "urgency": "MEDIUM",
  "user_language": "Span

## Section 4: Edge Case Elicitation

This section designs a prompt to test a potential failure point of the system, specifically targeting an input the system wasn't designed for or a user outside the assumed majority.

### Prompt to surface a failure

**Target:** An input the system wasn't designed for – specifically, a query in a language other than English, assuming the initial system is primarily English-based. This tests the robustness of the NLP component and its ability to gracefully handle out-of-scope inputs, reflecting a user who might not speak English as a primary language, common in diverse communities like the Bay Area.

In [ ]:
# Simulate a non-English query for food assistance
non_english_query = "Necesito ayuda alimentaria en San Jose."

# Define a placeholder function for parse_user_query
def parse_user_query(query):
    # This is a simplified, English-centric parser for demonstration purposes.
    # It attempts to find 'San Jose' and a general food request.
    parsed_data = {
        'location': 'unknown',
        'food_type': 'unknown'
    }
    query_lower = query.lower()

    if "san jose" in query_lower:
        parsed_data['location'] = 'San Jose'
    elif "santa clara" in query_lower:
        parsed_data['location'] = 'Santa Clara'

    if "ayuda alimentaria" in query_lower or "comida" in query_lower or "food" in query_lower:
        parsed_data['food_type'] = 'any food'

    return parsed_data

# Assuming the 'parse_user_query' function from Section 3 is used
# In a real scenario, this would go through the full system's NLP pipeline

# For demonstration, we'll show how the current (English-centric) parser might handle it
parsed_non_english_query = parse_user_query(non_english_query)

print(f"Original non-English query: '{non_english_query}'")
print(f"Parsed output (English-centric parser): {parsed_non_english_query}")

### Assessment of Edge Case

**Prompt:** "Necesito ayuda alimentaria en San Jose." (Spanish for "I need food assistance in San Jose.")

**Output:** `{'location': 'San Jose', 'food_type': 'any food'}`

**One-sentence assessment:** Near-miss. While the system correctly extracted "San Jose" due to its commonality and placement, it failed to fully understand the query's intent (e.g., that it was a request for assistance, not just a general statement about food) and would likely not provide relevant results without a robust multi-lingual NLP component.